# Notebook 13: Sliding Window Inference (predict.py)

**Module:** 12 Capstone — water-bodies-detection  
**Duration:** ~3 hrs

---

## Learning Objectives

1. Understand sliding window over full mosaic
2. Implement Hann window blending concept
3. Apply TTA d4 (8 dihedral transforms)
4. Match train normalization at inference

## predict.py Overview

Full GeoTIFF too large for GPU → **sliding window** inference.

```
1. Read mosaic with rasterio
2. Generate overlapping 512×512 windows (50% overlap)
3. For each window:
   a. Select bands [2,3,4,6,7,8]
   b. robust_normalize_tile (same as training)
   c. Model forward → logits (2, H, W)
   d. Optional TTA d4 (8 views averaged)
   e. Sigmoid → probabilities
4. Hann-weighted blend into full raster
5. Write *_aqua.tif + *_boundary.tif (float32)
```

In [ ]:
import os, json, yaml
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (8, 5)
REPO = '../../water-bodies-detection'
print('Capstone repo path:', os.path.abspath(REPO) if os.path.exists(REPO) else 'clone water-bodies-detection alongside Machine-Learning')

In [ ]:
def hann_blend_weights(h, w, floor=1e-3):
    wy = np.hanning(h)
    wx = np.hanning(w)
    return np.outer(wy, wx) + floor

w = hann_blend_weights(512, 512)
plt.imshow(w, cmap='viridis'); plt.title('Hann blend weights (center weighted)'); plt.colorbar(); plt.show()

## TTA d4

8 transforms: 4 rotations × 2 (with/without flip).

Each view: transform input → predict → inverse transform output → average.

**Cost:** 8× forward passes — use for production quality, skip for speed.

## blend_weights_2d

Hann window reduces seam artifacts at tile edges — center pixels weighted higher than edges.

---

## Summary

Sliding window + Hann blending + TTA = seamless full-mosaic predictions.

**Next:** [14_Probability_Rasters.ipynb](14_Probability_Rasters.ipynb)